[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C63_ML_System_Design_Course/05_case_studies/05_case_studies.ipynb)

# 05 · 案例库：六个完整设计演练（自评分器 / 关键数字速查 / 设计文档模板）

目标：把六个案例的演练结果，从「感觉讲得还不错」变成**可以量化、可以对比的分数**。

本 notebook 你会亲手实现：
1. **六案例关键数字速查表** —— 每个案例的量级参考（QPS/延迟/参数量/成本），内置为数据结构
2. **通用脚手架自检器** —— 开场 90 秒覆盖度检查、45 分钟时间条时间分配诊断
3. **澄清问题覆盖度检查器** —— 给定你实际问的问题，对照清单找出漏问的维度
4. **追问覆盖度检查器** —— 给定你准备的追问方向，对照该案例的必备追问集合
5. **案例演练综合自评分器** —— 把以上几项加权合成一个总分，附带「提分收益」排序
6. **可复用设计文档模板生成器** —— 一份能直接套用到任何案例的结构化文档骨架

> 心智模型：**六个案例练的是「同一套骨架能不能在不同题面下重新长出来」，
> 自评分器要检查的正是「骨架的每一格有没有被真正填上」，而不是「你讲了多少字」。**

## 0 · 环境自检

本课全程只用标准库 + numpy。没有 GPU 依赖、不联网、不下载数据。

In [ ]:
import sys, math
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)

assert sys.version_info >= (3, 8), '需要 Python 3.8+'
print('\n✅ 环境自检通过：本课不需要 GPU、不需要联网。')

## 1 · 六案例关键数字速查表

不要求精确，要求**量级不错**——面试里被问到「大概什么规模」时，能立刻给出一个数量级答案。

In [ ]:
# 六个案例的量级速查：QPS / 端到端延迟预算(ms) / 模型参数量级(M) / 成本量级(元/千次)
# 数字均为教学用的合理量级近似，不是任何真实系统的实测值
CASE_NUMBERS = {
    'tsr':        dict(qps=300,  latency_ms=30,  params_m=8,   cost_per_1k=0.05, note='车端在线为主'),
    'lane':       dict(qps=300,  latency_ms=20,  params_m=15,  cost_per_1k=0.06, note='车端在线，频率要求更高'),
    'retrieval':  dict(qps=50,   latency_ms=100, params_m=100, cost_per_1k=0.50, note='ANN索引查询，云端在线'),
    'moderation': dict(qps=1000, latency_ms=200, params_m=300, cost_per_1k=1.00, note='近线为主，含人工复核队列'),
    'ranking':    dict(qps=5000, latency_ms=50,  params_m=50,  cost_per_1k=0.10, note='云端在线，高QPS低延迟'),
    'platform':   dict(qps=None, latency_ms=None, params_m=0,  cost_per_1k=0.00, note='离线/近线批处理，无实时QPS概念'),
}
CASE_NAMES = {
    'tsr': 'TSR感知系统', 'lane': '车道线/可行驶区域', 'retrieval': '图搜图/场景检索',
    'moderation': '内容审核/异常检测', 'ranking': '推荐排序', 'platform': '数据闭环平台',
}

assert len(CASE_NUMBERS) == 6
assert set(CASE_NUMBERS) == set(CASE_NAMES)
# 合理性检查：内容审核的QPS（互联网规模）应远高于单车队TSR的QPS
assert CASE_NUMBERS['moderation']['qps'] > CASE_NUMBERS['tsr']['qps']
# 推荐排序QPS应是六案例中最高量级（互联网大规模在线服务）
assert CASE_NUMBERS['ranking']['qps'] == max(v['qps'] for v in CASE_NUMBERS.values() if v['qps'] is not None)
# 数据闭环平台没有实时QPS概念（批处理系统）
assert CASE_NUMBERS['platform']['qps'] is None

print(f"{'案例':<20}{'QPS':>8}{'延迟(ms)':>10}{'参数量(M)':>12}{'成本/千次':>12}  备注")
for k, v in CASE_NUMBERS.items():
    qps_s = str(v['qps']) if v['qps'] is not None else 'N/A(批处理)'
    lat_s = str(v['latency_ms']) if v['latency_ms'] is not None else 'N/A'
    print(f"{CASE_NAMES[k]:<20}{qps_s:>8}{lat_s:>10}{v['params_m']:>12}{v['cost_per_1k']:>12.2f}  {v['note']}")
print('\n✅ 六案例速查表就位。')

## 2 · 通用脚手架自检器：开场 90 秒 + 45 分钟时间条

第一步：检查开场 90 秒是否覆盖了「复述/澄清/路线图」三个动作。

In [ ]:
OPENING_ACTIONS = ['restate', 'clarify', 'roadmap']   # 90秒开场的三个必备动作

def opening_check(actions_done):
    """actions_done: 实际做了的动作集合（子集）。返回缺失的动作列表（保持固定顺序）。"""
    return [a for a in OPENING_ACTIONS if a not in actions_done]

assert opening_check({'restate', 'clarify', 'roadmap'}) == []
assert opening_check({'restate'}) == ['clarify', 'roadmap']
assert opening_check(set()) == OPENING_ACTIONS
print('完整开场 ->', opening_check({'restate', 'clarify', 'roadmap'}))
print('只复述了题目 ->', opening_check({'restate'}), ' ← 少了这两步，面试官不知道接下来45分钟怎么走')
print('\n✅ 开场检查器就位。')

## 3 · 七步时间分配诊断器

45 分钟七步版预算（与本模块第 1 节的时间条一致）：范围5 + 指标5 + 数据10 + 建模10 + 评测8 + 部署5 + 风险2 = 45。

In [ ]:
SEVEN_STEP_BUDGET = {
    'scope': 5, 'metrics': 5, 'data': 10, 'modeling': 10,
    'eval': 8, 'serving': 5, 'risk': 2,
}
assert sum(SEVEN_STEP_BUDGET.values()) == 45

def time_allocation_score(actual, recommended=SEVEN_STEP_BUDGET, total=45):
    """actual: {step: 实际花的分钟数}。用绝对偏差之和归一化到 [0,1]，偏差越大分越低。
    偏差满分扣分点：偏差总和达到 2×total 时得分清零（此时几乎每一步都严重跑偏）。"""
    dev = sum(abs(actual.get(k, 0) - recommended[k]) for k in recommended)
    return round(max(0.0, 1 - dev / (2 * total)), 3)

perfect = dict(SEVEN_STEP_BUDGET)
s_perfect = time_allocation_score(perfect)
assert s_perfect == 1.0, s_perfect

# 常见翻车：建模讲了30分钟，评测和部署几乎没讲（呼应本模块第1节的CALLOUT danger）
overrun_modeling = dict(scope=5, metrics=3, data=5, modeling=30, eval=1, serving=1, risk=0)
s_bad = time_allocation_score(overrun_modeling)
assert 0.0 <= s_bad < 0.6, s_bad
print(f'理想时间分配 -> 得分 {s_perfect}')
print(f'建模超时(30min)、评测部署几乎没讲 -> 得分 {s_bad}')
print('\n✅ 时间分配诊断器就位：这就是「把90%时间花在建模上」在数字上的样子。')

## 4 · 澄清问题覆盖度检查器

C65-04 的澄清清单模板：目标 / 用户 / 约束 / 现状 / 成功标准 / 不做什么。六个字段是通用骨架，本模块每个案例的
「澄清问题」小节都是这六个字段在具体题面下的展开。

In [ ]:
CLARIFY_CHECKLIST = ['goal', 'user', 'constraint', 'status_quo', 'success_metric', 'non_goal']

def clarify_coverage(asked):
    """asked: 实际问到的维度集合。返回 (覆盖率, 缺失维度列表)。"""
    missing = [k for k in CLARIFY_CHECKLIST if k not in asked]
    coverage = 1 - len(missing) / len(CLARIFY_CHECKLIST)
    return round(coverage, 3), missing

cov1, miss1 = clarify_coverage(set(CLARIFY_CHECKLIST))
assert cov1 == 1.0 and miss1 == []

# TSR 案例常见的澄清问题只覆盖了 用户/约束/成功标准，漏了 目标/现状/不做什么
cov2, miss2 = clarify_coverage({'user', 'constraint', 'success_metric'})
assert abs(cov2 - 0.5) < 1e-9, cov2
assert set(miss2) == {'goal', 'status_quo', 'non_goal'}, miss2
print(f'全覆盖 -> 覆盖率 {cov1}, 缺失 {miss1}')
print(f'只问了三个维度 -> 覆盖率 {cov2}, 缺失 {miss2}')
print('\n✅ 澄清覆盖度检查器就位：「不做什么」这一维度最容易被漏问，恰恰是加分项。')

## 5 · 追问覆盖度检查器：六案例各自的必备追问集合

In [ ]:
# 每个案例的「必备追问方向」（不要求原话，只要求方向被覆盖到）——对应本模块每个案例的「追问预案」表
REQUIRED_FOLLOWUPS = {
    'tsr':        {'error_attribution', 'longtail_data', 'offline_online_consistency'},
    'lane':       {'calibration_drift', 'lane_missing_fallback'},
    'retrieval':  {'index_consistency', 'embedding_eval'},
    'moderation': {'threshold_from_cost', 'adversarial_drift'},
    'ranking':    {'offline_online_gap', 'position_bias'},
    'platform':   {'budget_priority', 'noise_vs_hardcase'},
}

def followup_gap(case_key, prepared):
    """prepared: 你实际准备了的追问方向集合。返回按字典序排序的缺失方向列表。"""
    required = REQUIRED_FOLLOWUPS[case_key]
    return sorted(required - set(prepared))

assert followup_gap('tsr', REQUIRED_FOLLOWUPS['tsr']) == []
gap = followup_gap('tsr', {'error_attribution'})
assert gap == ['longtail_data', 'offline_online_consistency'], gap
gap2 = followup_gap('moderation', set())
assert gap2 == ['adversarial_drift', 'threshold_from_cost'], gap2

for key in REQUIRED_FOLLOWUPS:
    print(f'{key:<12} 必备追问方向: {sorted(REQUIRED_FOLLOWUPS[key])}')
print('\n✅ 追问覆盖度检查器就位。')

## 6 · 案例演练综合自评分器

把「开场」「时间分配」「澄清覆盖度」「追问覆盖度」「关键取舍是否显式说出」五个维度加权合成总分，
权重设计参照 C63-00 的评分卡精神：**沟通与风险意识的权重不低于技术深度**。

In [ ]:
DRILL_RUBRIC = [
    ('opening',        '开场完整度',   0.15),
    ('time_alloc',     '时间分配合理度', 0.20),
    ('clarify_cov',    '澄清覆盖度',   0.20),
    ('tradeoff_explicit', '取舍显式度', 0.25),
    ('followup_cov',  '追问准备度',   0.20),
]
PASS_LINE = 3.5   # 满分5分制

assert abs(sum(w for _, _, w in DRILL_RUBRIC) - 1.0) < 1e-12

def drill_score(sub_scores_0to1):
    """sub_scores_0to1: {维度key: 0~1的子分数}。转换成5分制加权总分。"""
    return round(sum(w * sub_scores_0to1[k] * 5 for k, _, w in DRILL_RUBRIC), 3)

# 案例A：只顾建模，时间分配差、取舍讲得含糊
weak = dict(opening=1.0, time_alloc=0.3, clarify_cov=0.5, tradeoff_explicit=0.2, followup_cov=0.3)
# 案例B：结构完整、取舍讲得清楚，但追问准备略弱
strong = dict(opening=1.0, time_alloc=0.9, clarify_cov=0.83, tradeoff_explicit=1.0, followup_cov=0.5)

sa, sb = drill_score(weak), drill_score(strong)
assert sa < PASS_LINE, sa
assert sb >= PASS_LINE, sb
print(f'案例A（只顾建模）  -> {sa:.2f}   ← 不及格')
print(f'案例B（结构完整+取舍清楚）-> {sb:.2f}   ← 通过')
print(f"\n取舍显式度权重最高({dict((k,w) for k,_,w in DRILL_RUBRIC)['tradeoff_explicit']})，"
      f'这与 C65-03「说清放弃了什么」的分量判断一致。')
print('\n✅ 综合自评分器就位。')

## ✏️ 练习 1：提分收益排序（复用五维评分卡）

实现 `drill_gap(sub_scores_0to1)`，返回 `[(维度中文名, 提升到满分的加权收益), ...]`，
收益 = `权重 × (1 - 当前子分数) × 5`，按收益**降序**排列，收益相同按维度 key 字典序升序。

In [ ]:
def drill_gap(sub_scores_0to1):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
gap = drill_gap(weak)
names = [n for n, _ in gap]
assert names[0] == '取舍显式度', names   # weak 的取舍显式度只有0.2，权重又最高，应排第一
vals = [v for _, v in gap]
assert all(v >= 0 for v in vals)
assert abs(sum(v for _, v in gap) + drill_score(weak) - 5.0) < 1e-9, '总收益+当前分应恰好等于满分5'
for n, v in gap:
    print(f'  {n:<10} 提到满分可加 {v:.2f}')
print('\n✅ 练习 1 通过：案例A该优先补的不是「多讲点技术细节」，是「把取舍显式说出来」。')

## ✏️ 练习 2：45 分钟时间条的超时预警器

实现 `budget_warning(elapsed, current_step)`，`current_step` 是当前正在讲的步骤 key（`SEVEN_STEP_BUDGET` 的 key）。
按累计预算表（`scope` 结束于第5分钟，`metrics`结束于第10分钟，以此类推）判断：

- 若 `elapsed` 超过「当前步骤应结束的时刻」不到 3 分钟 → `'ok'`
- 超过 3–8 分钟 → `'wrap_up'`（该收尾这一步了）
- 超过 8 分钟以上 → `'skip_ahead'`（直接跳到下一步，不要恋战）

In [ ]:
def budget_warning(elapsed, current_step):
    # TODO: 先算出 current_step 累计应结束的时刻，再按规则返回
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 累计结束时刻：scope=5, metrics=10, data=20, modeling=30, eval=38, serving=43, risk=45
assert budget_warning(4, 'scope') == 'ok'
assert budget_warning(6, 'scope') == 'ok'          # 5+1
assert budget_warning(10, 'scope') == 'wrap_up'    # 5+5
assert budget_warning(15, 'scope') == 'skip_ahead' # 5+10
assert budget_warning(30, 'modeling') == 'ok'      # 恰好卡在预算终点
assert budget_warning(35, 'modeling') == 'wrap_up' # 30+5
assert budget_warning(45, 'modeling') == 'skip_ahead'  # 30+15
for e, s in [(4, 'scope'), (10, 'scope'), (15, 'scope'), (35, 'modeling')]:
    print(f'第{e:>2}分钟，正在讲「{s}」 -> {budget_warning(e, s)}')
print('\n✅ 练习 2 通过：这就是白板旁那条时间带在数字上的样子。')

## ✏️ 练习 3：设计文档模板填充器

实现 `fill_design_doc(case_key, answers)`，`answers` 是 `{七步key: 内容字符串}` 的字典（key 同 `SEVEN_STEP_BUDGET`）。
返回一份 Markdown 字符串，**必须包含七个二级标题**（用案例中文名做标题，七步内容按 `SEVEN_STEP_BUDGET` 的顺序展开），
每个标题下面跟对应的内容（如果 `answers` 里没提供该步骤，写 `"(未填写)"`）。

标题格式：`## {步骤序号}. {步骤中文名}`，步骤中文名映射：
`{'scope':'范围','metrics':'指标','data':'数据','modeling':'建模','eval':'评测','serving':'部署','risk':'风险'}`

In [ ]:
STEP_CN = {'scope': '范围', 'metrics': '指标', 'data': '数据', 'modeling': '建模',
           'eval': '评测', 'serving': '部署', 'risk': '风险'}

def fill_design_doc(case_key, answers):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
doc = fill_design_doc('tsr', {'scope': '只做2D框，不做信号灯', 'modeling': 'baseline优先，见C55-02'})
assert doc.count('##') == 7, doc.count('##')
assert '1. 范围' in doc and '只做2D框，不做信号灯' in doc
assert '4. 建模' in doc and 'C55-02' in doc
assert '2. 指标' in doc and '(未填写)' in doc   # 没提供metrics，应回退到占位符
assert CASE_NAMES['tsr'] in doc or 'tsr' in doc  # 允许用案例key或中文名标注案例本身
print(doc[:300], '...\n')
print('\n✅ 练习 3 通过：这是六个案例现场演练时可以直接照抄的骨架。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def drill_gap(sub_scores_0to1):
    out = [(name, w * (1 - sub_scores_0to1[k]) * 5) for k, name, w in DRILL_RUBRIC]
    return sorted(out, key=lambda kv: (-kv[1], kv[0]))

In [ ]:
# 练习 2 参考答案
def budget_warning(elapsed, current_step):
    order = list(SEVEN_STEP_BUDGET)
    cum = {}
    acc = 0
    for k in order:
        acc += SEVEN_STEP_BUDGET[k]
        cum[k] = acc
    deadline = cum[current_step]
    over = elapsed - deadline
    if over < 3:
        return 'ok'
    if over < 8:
        return 'wrap_up'
    return 'skip_ahead'

In [ ]:
# 练习 3 参考答案
def fill_design_doc(case_key, answers):
    lines = [f'# 设计文档：{CASE_NAMES.get(case_key, case_key)}', '']
    for i, k in enumerate(SEVEN_STEP_BUDGET, 1):
        lines.append(f'## {i}. {STEP_CN[k]}')
        lines.append(answers.get(k, '(未填写)'))
        lines.append('')
    return '\n'.join(lines)

---
## 🧪 真实工程胶囊：可复用设计文档模板（照抄到面试前的准备笔记里）

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# ML System Design 通用设计文档模板 —— 45分钟演练前打印在纸上
# ══════════════════════════════════════════════════════════════════════

# 【题面】（面试官原话，一字不改地抄下来，避免自己曲解题意）
#   ...

# 【① 范围】(5min)
#   用户是谁：
#   不做什么：
#   硬约束（延迟/算力/合规）：

# 【② 指标】(5min)
#   业务指标 -> 模型指标 -> 系统指标：
#   代价矩阵（FN代价 vs FP代价）：

# 【③ 数据】(10min)
#   来源与标注：
#   长尾/不平衡处理：
#   泄漏防范（时间/地理/设备/track级）：

# 【④ 建模】(10min)
#   baseline方案：
#   核心难点与对应解法（只引用结论，标注"见CXX-XX"，不重述机制）：

# 【⑤ 评测】(8min)
#   离线切片方案：
#   在线A/B设计：
#   离线-在线一致性风险：

# 【⑥ 部署】(5min)
#   容量估算（QPS/显存/算力/成本，量级即可）：
#   延迟预算分解 + 关键路径：
#   降级方案（模型/特征/兜底）：

# 【⑦ 风险】(2min)
#   最大的未知数是什么：
#   如果有更多时间会先验证什么：

# ══════════════════════════════════════════════════════════════════════
# 收尾三件套（最后5分钟，不要省略）
# ══════════════════════════════════════════════════════════════════════
# 1) 结论先行复盘取舍：「如果只能选一个方案，我会选X，因为...；放弃的是Y，代价是...」
# 2) 主动列风险与下一步：「给我更多时间，我会重点验证...」
# 3) 反问一个真实问题：「这个系统实际的延迟瓶颈现在在哪一层？」

# ══════════════════════════════════════════════════════════════════════
# 六案例速记（各自最容易被扣分的第一个坑）
# ══════════════════════════════════════════════════════════════════════
# ① TSR         : 只讲模型不讲数据闭环和降级方案
# ② 车道线      : 把"车道线"和"可行驶区域"当成同一个任务
# ③ 图搜图      : 不区分"像素相似"和"语义相似"两种检索
# ④ 内容审核    : 用accuracy衡量极度不平衡的问题
# ⑤ 推荐排序    : 不知道离线AUC涨点可能是position bias在撒谎
# ⑥ 数据闭环平台: 讲成技术清单而不是接口/门禁设计
'''
print(RECIPE)
for token in ['题面', '范围', '指标', '数据', '建模', '评测', '部署', '风险', '收尾三件套', 'position bias']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：七步骨架 + 收尾三件套 + 六案例速记 —— 可直接打印带进模拟面试')

### 小结

- **本模块是全课落点**：六个案例把模块 00–04 的方法论缝合成六段可以照着走一遍的完整 45 分钟演练，
  但**它们不是要背下来的剧本**——目标是练出「同一套骨架能不能在没见过的题面上重新长出来」的能力。
- **通用脚手架**：开场 90 秒（复述→澄清→路线图）、白板四象限（范围指标 / 架构图 / 数据评测 / 容量风险）、
  45 分钟七步时间条（范围5 建模最长10 评测8 部署5 风险2），**最容易的翻车是把大半时间耗在建模上**。
- **① TSR 是唯一必须练到脱稿的案例**：级联召回是乘法关系（$R_{system}\approx R_{det}\times Acc_{cls}$），
  数据/评测/部署的技术细节全部只引用 C55/C57/C58/C60 的结论，不重述——这本身就是评分点。
- **六个案例的共性远大于差异**：七步框架、三层指标映射、代价不对称工作点、服务容量心算是可迁移骨架，
  领域知识只是换皮——②③④⑤⑥各自的「第一个坑」都值得单独记住（见工程胶囊速记）。
- **有限时间的准备优先级**：① 精通到脱稿 → ② 挑 1-2 个贴近背景的案例做第二重点 → ③ 其余案例做到"认出模式"即可，
  不要对六个案例平均分配时间。

—— 至此，C63《ML 系统设计面试》全课完结。搭配 C61（检测专项白板题）、C64（技术知识问答）、
C65（结构化问题求解与沟通），四门课覆盖了 XPENG TSR 一面 HR 说明里的全部三个板块。